# 离线推理全流程 — ATC 编译与 ACL 推理

在前面的章节中，我们了解了 GE 的全栈定位、AscendIR 与 ES 构图的基础概念。本节聚焦 GE 的**离线路径**：先把计算图编译成 OM 离线模型（ATC / Build 接口），再用 ACL 在昇腾设备上加载并执行推理。整条链路与昇腾产品的"训练好模型 → 离线部署"工程实践完全一致。

我们以一个最小的 **Add 算子图**贯穿全节：两个 `(2,3)` 的 float 输入相加。这样无论看 ATC 命令、Build 接口、还是 ACL 执行，关注点都集中在"流程"本身，而不被复杂网络干扰。

本节学习大纲：

1. **ATC 基础用法** —— 命令行 / Parser 接口 / 离线编译四步流程
2. **输入输出配置** —— 命令行参数与 `build_options`
3. **OM 产物理解** —— OM 里到底装了什么，回顾图编译四阶段
4. **ACL 模型加载** —— `aclmdlLoadFromFile` 等三种加载方式
5. **ACL 模型执行** —— 同步 `aclmdlExecute` 与异步 `aclmdlExecuteAsync`
6. **输入输出管理** —— `aclmdlDesc` 描述 + `aclmdlDataset` 数据集
7. **动手实践** —— 离线编译执行全流程（numpy 对拍基准 + Python 可运行源码 + C++ 参考）
8. **课后练习**

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>贯穿全节的 Add 样例：<code>x = [[1,2,3],[4,5,6]]</code>，<code>y = [[10,20,30],[40,50,60]]</code>，期望输出 <code>[[11,22,33],[44,55,66]]</code>。后面的 numpy 对拍 cell 会真实算出这个结果，作为推理正确性的基准。</p>
</blockquote>
</div>

## 1. ATC 基础用法

ATC（Ascend Tensor Compiler）是 GE 的离线编译工具，把 ONNX / TensorFlow / Caffe 等框架模型，或代码里构建的 Graph，编译成昇腾设备可执行的模型数据；在离线部署场景中，这份 GeModel 会被序列化并落盘为 **OM（Offline Model）** 文件。编译过程纯在 Host 侧完成，**不需要昇腾设备在线**（无卡也能编，只要用 `--soc_version` 指定目标芯片）。

GE 提供三条等价的离线编译入口：**命令行 ATC**、**Parser 解析框架模型**、**Build 接口编译代码内的 Graph**。

<p align="left"><img src="./images/compile_api_flow.svg" alt="离线编译入口与产物流转" width="80%"></p>

### 1.1 命令行编译

对已经导出的 ONNX/PB 模型文件，一条 `atc` 命令即可生成 OM：

```bash
atc --model=$HOME/module/resnet50.onnx \
    --framework=5 \
    --output=$HOME/module/out/onnx_resnet50 \
    --soc_version=<soc_version>
```

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>参数</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td><code>--model</code></td><td>输入模型文件路径（.onnx / .pb / .prototxt 等）</td></tr>
<tr><td><code>--framework</code></td><td>原始框架：<strong>3</strong>=TensorFlow，<strong>5</strong>=ONNX，<strong>1</strong>=MindSpore air</td></tr>
<tr><td><code>--output</code></td><td>输出 OM 文件名（<strong>不含</strong> <code>.om</code> 后缀，工具会自动补）</td></tr>
<tr><td><code>--soc_version</code></td><td>目标芯片版本，如 <code>Ascend910B1</code></td></tr>
</tbody>
</table>
</div>

`--soc_version` 的取值在安装了 AI 处理器的服务器上执行 **`npu-smi info`** 查询 **Name** 字段获得（例如 Name 为 `xxxyy`，则配置值为 `Ascendxxxyy`）。

编译成功时，命令行会打印：

```text
ATC run success, welcome to the next use.
```

### 1.2 Parser 接口：把框架模型解析成 Graph

如果想在代码里加载框架模型再做后续处理，GE 提供 Parser 接口，将模型文件解析为 GE 的 `Graph`：

<p align="left"><img src="./images/parser_graph.svg" alt="Parser 解析框架模型生成 Graph" width="72%"></p>

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>接口</th><th>功能</th></tr>
</thead>
<tbody>
<tr><td><code>aclgrphParseONNX</code></td><td>将 ONNX 模型文件解析为 Graph</td></tr>
<tr><td><code>aclgrphParseTensorFlow</code></td><td>将 TensorFlow 模型文件解析为 Graph</td></tr>
<tr><td><code>aclgrphParseCaffe</code></td><td>将 Caffe 模型文件解析为 Graph</td></tr>
<tr><td><code>aclgrphParseONNXFromMem</code></td><td>将<strong>加载到内存</strong>的 ONNX 模型解析为 Graph</td></tr>
</tbody>
</table>
</div>

解析得到的 `Graph` 可以直接喂给下面的 Build 接口编译，也可以先做图层面的修改。

### 1.3 离线编译流程

无论 Graph 来自 Parser 还是 ES 构图（见第 7 节动手实践），代码内编译都遵循固定的**四步**：

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>步骤</th><th>接口</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td>1. 初始化</td><td><code>aclgrphBuildInitialize</code></td><td>初始化编译环境、申请资源；无卡时在此传入 <code>ge.socVersion</code></td></tr>
<tr><td>2. 构建模型</td><td><code>aclgrphBuildModel</code></td><td>将 <code>Graph</code> 编译为适配 AI 处理器的模型数据，结果在内存缓冲区</td></tr>
<tr><td>3. 保存模型</td><td><code>aclgrphSaveModel</code></td><td>离线部署场景下，将 GeModel 序列化并落盘为 <code>.om</code> 文件</td></tr>
<tr><td>4. 资源释放</td><td><code>aclgrphBuildFinalize</code></td><td>释放编译资源</td></tr>
</tbody>
</table>
</div>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>上表为 C++ 接口；第 7 节动手实践会给出调用这四步的完整可运行 C++ 源码。Python 侧一一对应 <code>from ge.offline_compile import build_initialize, build_model, save_model, build_finalize</code>，完整 Python 工程见第 7.6 节链接。</p>
</blockquote>
</div>


## 2. 输入输出配置

编译时可以约束模型的输入 shape、输出类型等规格。命令行和代码两种方式都支持。

### 2.1 命令行配置

```bash
atc --model=resnet50.onnx \
    --framework=5 \
    --output=resnet50 \
    --soc_version=<soc_version> \
    --input_shape="input:1,3,224,224" \
    --output_type="output:FP32"
```

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>参数</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td><code>--input_shape</code></td><td>指定输入 shape，格式 <code>"输入名:N,C,H,W"</code>；动态 shape 用 <code>-1</code> 占位并配合 <code>--dynamic_*</code></td></tr>
<tr><td><code>--output_type</code></td><td>指定输出数据类型，格式 <code>"输出名:FP32"</code></td></tr>
<tr><td><code>--input_format</code></td><td>指定输入排布，如 <code>NCHW</code> / <code>ND</code></td></tr>
</tbody>
</table>
</div>

### 2.2 编译选项配置

在 Build 接口里，编译选项通过 `build_options` 传入。这正是本节 Add 样例真实使用的写法：

C++：

```cpp
std::map<ge::AscendString, ge::AscendString> build_options;
build_options.emplace(ge::AscendString("input_format"), ge::AscendString("ND"));
ge::aclgrphBuildModel(*graph, build_options, model);
```

Python：

```python
build_options = {"input_format": "ND"}
model = build_model(graph, build_options)
```

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>Add 图的两个输入都是 <code>(2,3)</code> 的 <code>ND</code> 张量，所以这里把 <code>input_format</code> 设为 <code>ND</code>。</p>
</blockquote>
</div>

## 3. OM 产物理解

OM（Offline Model）是 GE 离线编译场景的落盘产物：GE 先把 Graph 编译成设备可执行的 GeModel，再在离线部署路径中把 GeModel 序列化为 OM 文件。OM 是一个**自包含**的二进制文件，加载它就能在昇腾设备上跑，**不再依赖原始 ONNX/PB 文件或前端框架运行时**。

### 3.1 编译过程：图编译四阶段

OM 不是简单地"打包模型"，而是 Graph 经过完整图编译流水线后的离线文件形态。回顾图编译的四个阶段：

<p align="left"><img src="./images/ge_compile_flow.svg" alt="图编译四阶段与 GeModel/OM 关系" width="80%"></p>

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>阶段</th><th>做什么</th></tr>
</thead>
<tbody>
<tr><td>原始图</td><td>Parser/构图得到的初始计算图</td></tr>
<tr><td>图准备</td><td>形状推导、数据类型处理、常量折叠等准备工作</td></tr>
<tr><td>图拆分</td><td>按引擎/子图边界切分，分配到不同执行单元</td></tr>
<tr><td>图优化</td><td>算子融合、内存复用、UB 融合等性能优化</td></tr>
<tr><td>图编译</td><td>生成设备可执行的 GeModel；离线场景可进一步序列化为 OM 文件</td></tr>
</tbody>
</table>
</div>

### 3.2 OM 文件里装了什么

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>组成部分</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td>算子二进制</td><td>编译后的算子 kernel 代码，适配昇腾 AI Core</td></tr>
<tr><td>权重数据</td><td>模型参数（卷积核、BN 参数等）；Add 样例无权重</td></tr>
<tr><td>执行流信息</td><td>Task 序列、Stream 分配、内存编排方案</td></tr>
<tr><td>模型元信息</td><td>输入输出描述、shape、dtype、模型版本等</td></tr>
</tbody>
</table>
</div>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>注意：OM <strong>不包含</strong>原始模型文件（.onnx / .pb）。这是离线模型可独立分发的根本原因——它已经把计算固化成芯片可执行的形态。</p>
</blockquote>
</div>

### 3.3 OM 的使用约束

- OM 可在昇腾设备上通过 ACL API 直接加载执行，独立分发，不依赖前端框架。
- OM 与**目标芯片版本绑定**：为 Ascend910B1 编的 OM 不能直接拿到别的芯片跑，换芯片需重新编译。


## 4. ACL 模型加载

拿到 OM 之后，进入执行侧。ACL（Ascend Computing Language）提供在昇腾设备上加载、执行 OM 的运行时 API；Python 侧对应 **pyACL**（`import acl`）。

### 4.1 三种加载方式

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>接口</th><th>说明</th></tr>
</thead>
<tbody>
<tr><td><code>aclmdlLoadFromFile</code></td><td>从文件路径加载 OM，运行时自行管理工作内存（最常用）</td></tr>
<tr><td><code>aclmdlLoadFromMem</code></td><td>从内存缓冲区加载 OM</td></tr>
<tr><td><code>aclmdlLoadFromFileWithMem</code></td><td>从文件加载并由用户指定工作/权重内存（精细化内存管理）</td></tr>
</tbody>
</table>
</div>

三种加载接口都会返回一个 **`model_id`**，后续的执行、获取描述、卸载，全部通过这个 ID 操作。

### 4.2 加载流程（C++）

```cpp
// 1. 初始化 ACL 运行环境
aclInit(nullptr);
aclrtSetDevice(0);

// 2. 加载 OM 模型，拿到 model_id
uint32_t model_id = 0;
aclmdlLoadFromFile("add_sample.om", &model_id);

// 3. 获取模型描述信息（输入输出规格）
aclmdlDesc *model_desc = aclmdlCreateDesc();
aclmdlGetDesc(model_desc, model_id);
```

### 4.3 加载流程（Python / pyACL）

```python
import acl
acl.init()
acl.rt.set_device(0)

model_id, ret = acl.mdl.load_from_file("add_sample.om")
model_desc = acl.mdl.create_desc()
acl.mdl.get_desc(model_desc, model_id)
```

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>pyACL 接口普遍返回 <code>(结果, ret)</code> 二元组，<code>ret == 0</code>（<code>ACL_SUCCESS</code>）才表示成功——这是 pyACL 的固定约定。</p>
</blockquote>
</div>

## 5. ACL 模型执行

加载完成后，通过 ACL 执行推理。ACL 提供**同步**和**异步**两种执行接口。

### 5.1 同步执行

```cpp
aclmdlExecute(model_id, input_dataset, output_dataset);
```

- 调用**返回时推理已完成**，输出 dataset 里就是结果。
- 编程简单，适合单次推理 / 教学场景。本节 Add 样例用的就是同步执行。

Python：

```python
acl.mdl.execute(model_id, input_dataset, output_dataset)
```

### 5.2 异步执行

```cpp
aclmdlExecuteAsync(model_id, input_dataset, output_dataset, stream);
aclrtSynchronizeStream(stream);   // 等待该 stream 上的任务全部完成
```

- 调用**立即返回**，推理可能尚未完成，必须 `aclrtSynchronizeStream` 等流同步后才能读输出。
- 配合 stream 使用，可让**计算与数据搬运重叠**，适合追求吞吐的高性能场景。

### 5.3 完整执行流程

参考 ge-document 中“应用开发 / 模型推理 / 静态 Shape 输入模型推理”的模型加载和模型执行流程，完整 ACL 推理主线如下：

<p align="left"><img src="./images/acl_execute_flow.svg" alt="ACL 模型加载与执行流程" width="32%"></p>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>同步 vs 异步的核心区别：<strong>同步接口返回即完成；异步接口返回后还要 <code>aclrtSynchronizeStream</code> 才能保证结果就绪。</strong></p>
</blockquote>
</div>


## 6. 输入输出管理

ACL 用两个对象管理推理的输入输出：`aclmdlDesc`（描述模型规格）和 `aclmdlDataset`（承载实际数据）。

### 6.1 aclmdlDesc —— 模型规格描述

通过 `aclmdlGetDesc` 拿到模型规格后，可查询：

- 输入个数 `aclmdlGetNumInputs` / 输出个数 `aclmdlGetNumOutputs`
- 每个输入输出的 size（字节）`aclmdlGetInputSizeByIndex` / `aclmdlGetOutputSizeByIndex`
- 每个输出的维度 `aclmdlGetOutputDims`

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>关键：<strong>输入输出 buffer 该申请多大，是从 <code>aclmdlDesc</code> 查出来的</strong>，不要硬编码——这样换模型时代码不用改。</p>
</blockquote>
</div>

### 6.2 aclmdlDataset —— 数据集

`aclmdlDataset` 是一组 `aclDataBuffer` 的容器，每个 buffer 指向一块 Device 内存：

```cpp
aclmdlDataset *dataset = aclmdlCreateDataset();
void *dev = nullptr;
aclrtMalloc(&dev, buf_size, ACL_MEM_MALLOC_NORMAL_ONLY);   // 按 desc 查出的 size 申请
aclDataBuffer *buffer = aclCreateDataBuffer(dev, buf_size);
aclmdlAddDatasetBuffer(dataset, buffer);                   // 加入数据集
```

### 6.3 完整输入输出流程

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>步骤</th><th>操作</th><th>接口</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>查询输入输出规格</td><td><code>aclmdlGetDesc</code> + <code>aclmdlGetInput/OutputSizeByIndex</code></td></tr>
<tr><td>2</td><td>按规格申请 Device 内存</td><td><code>aclrtMalloc</code></td></tr>
<tr><td>3</td><td>输入数据 Host→Device 拷贝</td><td><code>aclrtMemcpy(..., ACL_MEMCPY_HOST_TO_DEVICE)</code></td></tr>
<tr><td>4</td><td>构造 Dataset 并挂入 buffer</td><td><code>aclmdlCreateDataset</code> / <code>aclCreateDataBuffer</code> / <code>aclmdlAddDatasetBuffer</code></td></tr>
<tr><td>5</td><td>执行推理</td><td><code>aclmdlExecute</code></td></tr>
<tr><td>6</td><td>输出结果 Device→Host 拷贝</td><td><code>aclrtMemcpy(..., ACL_MEMCPY_DEVICE_TO_HOST)</code></td></tr>
<tr><td>7</td><td>释放 Dataset / Device 内存</td><td><code>aclmdlDestroyDataset</code> / <code>aclrtFree</code></td></tr>
</tbody>
</table>
</div>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>第 7 节的 <code>common.cpp</code> 把第 1~7 步真实实现了一遍，可以对照阅读。</p>
</blockquote>
</div>

## 7. 动手实践：离线编译执行全流程

下面把上面所有概念串成一个可运行的 Add 样例，**与前几节一致采用 C++**。组织方式：

1. **numpy 对拍基准**（任何环境可直接运行）：先用 numpy 算出 Add 的期望输出 `[[11,22,33],[44,55,66]]`，作为后续昇腾推理结果的比对基准。
2. **C++ 源码**（`%%writefile` 落盘）：与 GE 真实样例一致的 `common.h` / `common.cpp` / `build_add_model.cpp` / `run_add_model.cpp` 与 `CMakeLists.txt`，包含 ES 构图、离线编译四步、ACL 加载执行。
3. **编译运行命令**：在已装 CANN 的昇腾环境用 CMake 编译并执行；轻量 notebook 环境无 NPU/CANN 时仅作展示。
4. **Python 参考**：完整 Python 工程指向 GE 仓库 [examples/offline_compile_run/python](https://gitcode.com/cann/ge/tree/master/examples/offline_compile_run/python)。

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>重要前置：第 2、3 步<strong>需要在已安装 CANN（toolkit + ops）的昇腾环境</strong>执行（并已 <code>source set_env.sh</code>）。当前轻量 notebook 环境若无 NPU/CANN，<strong>只有第 1 个 numpy cell 能真正运行</strong>，其余 cell 仅把源码落盘 / 展示命令，不会真正编译或调用昇腾。</p>
</blockquote>
</div>

In [ ]:
# === 可直接运行：numpy 对拍基准（无任何昇腾依赖）===
# 用 numpy 复现 Add 图的语义，得到推理"应该"产出的结果，作为后续 NPU 推理的比对基准。
import numpy as np

# 与 GE 样例 create_sample_inputs() 完全一致的输入
input_x = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=np.float32)
input_y = np.array([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]], dtype=np.float32)

expected = input_x + input_y   # Add 图的语义就是逐元素相加
print("input_x =\n", input_x)
print("input_y =\n", input_y)
print("期望输出 expected =\n", expected)

# 断言期望输出确实是 [[11,22,33],[44,55,66]]
assert np.array_equal(expected, np.array([[11, 22, 33], [44, 55, 66]], dtype=np.float32))
print("\n[OK] 对拍基准就绪，期望输出 = [[11,22,33],[44,55,66]]")

### 7.1 公共逻辑（ES 构图 + ACL 数据集管理）

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p><strong>ES 构图（Eager Style）速览</strong>：一种函数风格的极简构图方式。它已在 <a href="../01_basic_concepts/01.03_ascend_ir.ipynb">1.3 节 · 7.3 ES 极简构图</a> 预览，完整接口（<code>EsGraphBuilder</code> / <code>CreateInput</code> / 运算符重载 / <code>BuildAndReset</code>）在 <a href="02.03_online_execution.ipynb">2.3 节 在线执行流程</a> 详述。读本节代码时，理解 <code>MakeOfflineAddGraph()</code> 用它把两个输入逐元素相加、构成一张 Add 图即可。</p>
</blockquote>
</div>

`common.h` 声明、`common.cpp` 实现本节复用的工具：

- `MakeOfflineAddGraph()` 用 **ES 构图**（`ge::es::EsGraphBuilder`）构建 Add 图——`CreateInput` 建输入、运算符重载 `input_x + input_y` 表达 Add、`SetOutput` 设输出、`BuildAndReset()` 产出 `ge::Graph`。
- `CreateIoDataset` / `CopyFloatInputs` / `PrintModelOutputs` 用 **ACL** 实现第 6 节那套"按 desc 查 size → aclrtMalloc → aclCreateDataBuffer → dataset"的输入输出管理；`SampleInputs()` 给出与 numpy 基准一致的 `{{1..6},{10..60}}`。

In [ ]:
%%writefile Sources/01.04/common.h
/* 公共头：IoBuffers 与 ES 构图 / ACL 数据集管理工具声明 */
#ifndef OFFLINE_ADD_SAMPLE_COMMON_H_
#define OFFLINE_ADD_SAMPLE_COMMON_H_

#include "acl/acl.h"
#include "acl/acl_mdl.h"
#include "ge/ge_api_error_codes.h"
#include "graph/graph.h"
#include <memory>
#include <vector>

constexpr int32_t kDeviceId = 0;

// 一组输入或输出在 Device 侧的 buffer 句柄集合
struct IoBuffers {
  aclmdlDataset *dataset = nullptr;
  std::vector<void *> device_ptrs;
  std::vector<size_t> sizes;
};

// ES 构图：构建两输入逐元素相加的 Add 图
std::unique_ptr<ge::Graph> MakeOfflineAddGraph();
// 样例输入 {{1..6},{10..60}}，与 numpy 基准一致
std::vector<std::vector<float>> SampleInputs();

// 按模型 desc 查 size → aclrtMalloc → dataset
ge::Status CreateIoDataset(aclmdlDesc *model_desc, bool is_input, IoBuffers *out);
// Host -> Device 拷贝输入
ge::Status CopyFloatInputs(const std::vector<std::vector<float>> &host_inputs, const IoBuffers &inputs);
// Device -> Host 拷回输出并打印
ge::Status PrintModelOutputs(aclmdlDesc *model_desc, const IoBuffers &outputs);

void ReleaseDataset(aclmdlDataset *dataset);
void TeardownAclSingleModelInfer(IoBuffers *input_io, IoBuffers *output_io, aclmdlDesc *model_desc, uint32_t model_id);

#endif  // OFFLINE_ADD_SAMPLE_COMMON_H_


In [ ]:
%%writefile Sources/01.04/common.cpp
/* 公共实现：与 GE 仓 examples/offline_compile_run/cpp/src/common.cpp 对齐（单模型精简版） */
#include "common.h"
#include "ge/es_graph_builder.h"
#include <iostream>

namespace {
constexpr size_t kFloatsPerLine = 3U;

// 把单个输出 buffer 从 Device 拷回 Host 并打印（shape + data）
ge::Status PrintOneOutput(aclmdlDesc *model_desc, size_t index, aclDataBuffer *data_buffer) {
  void *data = aclGetDataBufferAddr(data_buffer);
  const size_t len = aclGetDataBufferSizeV2(data_buffer);
  if (data == nullptr || len == 0U) {
    std::cerr << "[Error] 输出 DataBuffer 无效, index=" << index << "\n";
    return ge::FAILED;
  }
  aclmdlIODims dims{};
  if (aclmdlGetOutputDims(model_desc, index, &dims) != ACL_SUCCESS) {
    std::cerr << "[Error] aclmdlGetOutputDims 失败\n";
    return ge::FAILED;
  }
  const size_t elem_count = len / sizeof(float);
  std::vector<float> host(elem_count);
  if (aclrtMemcpy(host.data(), len, data, len, ACL_MEMCPY_DEVICE_TO_HOST) != ACL_SUCCESS) {
    std::cerr << "[Error] aclrtMemcpy(output D2H) 失败\n";
    return ge::FAILED;
  }
  std::cout << "Output[" << (index + 1) << "] shape:";
  for (size_t d = 0; d < dims.dimCount; ++d) {
    std::cout << " " << dims.dims[d];
  }
  std::cout << "\n data:";
  for (size_t j = 0; j < elem_count; ++j) {
    std::cout << (j % kFloatsPerLine == 0U ? "\n " : " ") << host[j];
  }
  std::cout << std::endl;
  return ge::SUCCESS;
}
}  // namespace

std::unique_ptr<ge::Graph> MakeOfflineAddGraph() {
  auto graph_builder = std::make_unique<ge::es::EsGraphBuilder>("OfflineAddGraph");
  auto input_x = graph_builder->CreateInput(0, "input_x", ge::DT_FLOAT, ge::FORMAT_ND, {2, 3});
  auto input_y = graph_builder->CreateInput(1, "input_y", ge::DT_FLOAT, ge::FORMAT_ND, {2, 3});
  auto result = input_x + input_y;        // 运算符重载表达 Add
  (void)graph_builder->SetOutput(result, 0);
  return graph_builder->BuildAndReset();
}

std::vector<std::vector<float>> SampleInputs() {
  return {{1.0F, 2.0F, 3.0F, 4.0F, 5.0F, 6.0F}, {10.0F, 20.0F, 30.0F, 40.0F, 50.0F, 60.0F}};
}

ge::Status CreateIoDataset(aclmdlDesc *model_desc, bool is_input, IoBuffers *out) {
  *out = IoBuffers{};
  const size_t io_num = is_input ? aclmdlGetNumInputs(model_desc) : aclmdlGetNumOutputs(model_desc);
  out->dataset = aclmdlCreateDataset();
  if (out->dataset == nullptr) {
    std::cerr << "[Error] aclmdlCreateDataset 失败\n";
    return ge::FAILED;
  }
  for (size_t i = 0; i < io_num; ++i) {
    const size_t buf_size =
        is_input ? aclmdlGetInputSizeByIndex(model_desc, i) : aclmdlGetOutputSizeByIndex(model_desc, i);
    void *dev = nullptr;
    if (aclrtMalloc(&dev, buf_size, ACL_MEM_MALLOC_NORMAL_ONLY) != ACL_SUCCESS) {  // size 从 desc 查，不硬编码
      std::cerr << "[Error] aclrtMalloc 失败\n";
      return ge::FAILED;
    }
    aclDataBuffer *data_buffer = aclCreateDataBuffer(dev, buf_size);
    (void)aclmdlAddDatasetBuffer(out->dataset, data_buffer);
    out->device_ptrs.push_back(dev);
    out->sizes.push_back(buf_size);
  }
  return ge::SUCCESS;
}

ge::Status CopyFloatInputs(const std::vector<std::vector<float>> &host_inputs, const IoBuffers &inputs) {
  for (size_t i = 0; i < host_inputs.size(); ++i) {
    const size_t bytes = host_inputs[i].size() * sizeof(float);
    if (aclrtMemcpy(inputs.device_ptrs[i], inputs.sizes[i], host_inputs[i].data(), bytes,
                    ACL_MEMCPY_HOST_TO_DEVICE) != ACL_SUCCESS) {
      std::cerr << "[Error] aclrtMemcpy(input) 失败\n";
      return ge::FAILED;
    }
  }
  return ge::SUCCESS;
}

ge::Status PrintModelOutputs(aclmdlDesc *model_desc, const IoBuffers &outputs) {
  const size_t num_buffers = aclmdlGetDatasetNumBuffers(outputs.dataset);
  for (size_t i = 0; i < num_buffers; ++i) {
    aclDataBuffer *data_buffer = aclmdlGetDatasetBuffer(outputs.dataset, i);
    if (PrintOneOutput(model_desc, i, data_buffer) != ge::SUCCESS) {
      return ge::FAILED;
    }
  }
  return ge::SUCCESS;
}

void ReleaseDataset(aclmdlDataset *dataset) {
  if (dataset == nullptr) {
    return;
  }
  const size_t num = aclmdlGetDatasetNumBuffers(dataset);
  for (size_t i = 0; i < num; ++i) {
    aclDataBuffer *buf = aclmdlGetDatasetBuffer(dataset, i);
    if (buf != nullptr) {
      (void)aclrtFree(aclGetDataBufferAddr(buf));
      (void)aclDestroyDataBuffer(buf);
    }
  }
  (void)aclmdlDestroyDataset(dataset);
}

void TeardownAclSingleModelInfer(IoBuffers *input_io, IoBuffers *output_io, aclmdlDesc *model_desc, uint32_t model_id) {
  if (input_io != nullptr) {
    ReleaseDataset(input_io->dataset);
  }
  if (output_io != nullptr) {
    ReleaseDataset(output_io->dataset);
  }
  if (model_desc != nullptr) {
    (void)aclmdlDestroyDesc(model_desc);
  }
  if (model_id != 0U) {
    (void)aclmdlUnload(model_id);
  }
  (void)aclrtResetDevice(kDeviceId);
  (void)aclFinalize();
}


### 7.2 离线编译 build_add_model.cpp

`build_add_model.cpp` 调用 `ge::aclgrph*` 的四个接口，对应第 1.3 节的"初始化 → 构建 → 保存 → 释放"，最终生成 `add_sample.om`。无卡编译时用 `--soc-version Ascend910B1` 指定目标芯片。

In [ ]:
%%writefile Sources/01.04/build_add_model.cpp
/* 离线编译四步：Initialize -> BuildModel -> SaveModel -> Finalize，产出 add_sample.om */
#include "common.h"
#include "ge/ge_ir_build.h"
#include <iostream>
#include <map>
#include <string>

namespace {
std::map<ge::AscendString, ge::AscendString> GlobalOptionsWithSoc(const std::string &soc_version) {
  std::map<ge::AscendString, ge::AscendString> opts;
  if (!soc_version.empty()) {  // 无卡编译时通过 ge.socVersion 指定目标芯片
    opts.emplace(ge::AscendString("ge.socVersion"), ge::AscendString(soc_version.c_str()));
  }
  return opts;
}
}  // namespace

int main(int argc, char **argv) {
  std::string soc_version;
  if (argc >= 3 && std::string(argv[1]) == "--soc-version") {
    soc_version = argv[2];
  }

  auto global_options = GlobalOptionsWithSoc(soc_version);
  if (ge::aclgrphBuildInitialize(global_options) != ge::GRAPH_SUCCESS) {  // 步骤 1：初始化编译环境
    std::cerr << "[Error] aclgrphBuildInitialize 失败\n";
    return -1;
  }
  std::cout << "[Info] 系统初始化成功\n";

  auto graph = MakeOfflineAddGraph();  // ES 构图得到 Add 图
  ge::ModelBufferData model;
  std::map<ge::AscendString, ge::AscendString> build_options;
  build_options.emplace(ge::AscendString("input_format"), ge::AscendString("ND"));  // Add 图输入为 ND
  if (ge::aclgrphBuildModel(*graph, build_options, model) != ge::GRAPH_SUCCESS) {    // 步骤 2：编译为内存模型
    std::cerr << "[Error] aclgrphBuildModel 失败\n";
    ge::aclgrphBuildFinalize();
    return -1;
  }
  std::cout << "[Info] 模型构建成功，模型大小: " << model.length << " bytes\n";

  if (ge::aclgrphSaveModel("add_sample", model) != ge::GRAPH_SUCCESS) {  // 步骤 3：序列化保存为 add_sample.om
    std::cerr << "[Error] aclgrphSaveModel 失败\n";
    ge::aclgrphBuildFinalize();
    return -1;
  }
  std::cout << "[Info] 模型保存成功，add_sample.om 已生成在当前目录。\n";

  ge::aclgrphBuildFinalize();  // 步骤 4：释放编译资源
  std::cout << "[Info] 系统释放成功\n";
  return 0;
}


### 7.3 加载执行 run_add_model.cpp（ACL 同步推理）

`run_add_model.cpp` 用 ACL 走完"aclInit → aclrtSetDevice → aclmdlLoadFromFile → aclmdlGetDesc → 准备 dataset → aclmdlExecute → 取输出 → 释放"全流程，对应第 4~6 节。

In [ ]:
%%writefile Sources/01.04/run_add_model.cpp
/* ACL 同步推理：load -> get_desc -> dataset -> execute -> 取输出 -> 释放 */
#include "common.h"
#include "acl/acl.h"
#include "acl/acl_mdl.h"
#include "acl/acl_rt.h"
#include <iostream>

int main() {
  std::cout << "[Info] 初始化运行环境及数据\n";
  if (aclInit(nullptr) != ACL_SUCCESS) {
    std::cerr << "[Error] aclInit 失败\n";
    return -1;
  }
  if (aclrtSetDevice(kDeviceId) != ACL_SUCCESS) {
    std::cerr << "[Error] aclrtSetDevice 失败\n";
    (void)aclFinalize();
    return -1;
  }

  uint32_t model_id = 0;
  if (aclmdlLoadFromFile("add_sample.om", &model_id) != ACL_SUCCESS) {  // 加载 OM，拿 model_id
    std::cerr << "[Error] aclmdlLoadFromFile 失败\n";
    (void)aclrtResetDevice(kDeviceId);
    (void)aclFinalize();
    return -1;
  }
  aclmdlDesc *model_desc = aclmdlCreateDesc();
  if (aclmdlGetDesc(model_desc, model_id) != ACL_SUCCESS) {  // 取模型描述
    std::cerr << "[Error] aclmdlGetDesc 失败\n";
    TeardownAclSingleModelInfer(nullptr, nullptr, model_desc, model_id);
    return -1;
  }

  IoBuffers input_io;
  IoBuffers output_io;
  if (CreateIoDataset(model_desc, true, &input_io) != ge::SUCCESS ||  // 按 desc 准备输入/输出 dataset
      CreateIoDataset(model_desc, false, &output_io) != ge::SUCCESS) {
    TeardownAclSingleModelInfer(&input_io, &output_io, model_desc, model_id);
    return -1;
  }

  std::cout << "[Info] 执行模型推理\n";
  if (CopyFloatInputs(SampleInputs(), input_io) != ge::SUCCESS) {  // Host -> Device
    TeardownAclSingleModelInfer(&input_io, &output_io, model_desc, model_id);
    return -1;
  }
  if (aclmdlExecute(model_id, input_io.dataset, output_io.dataset) != ACL_SUCCESS) {  // 同步执行
    std::cerr << "[Error] aclmdlExecute 失败\n";
    TeardownAclSingleModelInfer(&input_io, &output_io, model_desc, model_id);
    return -1;
  }

  std::cout << "[Info] 模型推理结果:\n";
  (void)PrintModelOutputs(model_desc, output_io);  // Device -> Host 并打印，期望 [[11,22,33],[44,55,66]]

  TeardownAclSingleModelInfer(&input_io, &output_io, model_desc, model_id);
  std::cout << "[Info] 所有资源释放成功\n";
  return 0;
}


### 7.4 CMake 工程 CMakeLists.txt

C++ 需要显式链接 CANN 的图 / 编译 / ACL 动态库。`CMakeLists.txt` 从 `$ASCEND_HOME_PATH` 找头文件与 `lib64` 下的库，编译出两个可执行文件 `build_add_model`（离线编译）和 `run_add_model`（加载执行）。链接库与 GE 仓 `examples/offline_compile_run/cpp` 保持一致。

In [ ]:
%%writefile Sources/01.04/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(offline_add_sample)

set(ASCEND_INSTALL_PATH $ENV{ASCEND_HOME_PATH})
set(CMAKE_CXX_STANDARD 17)
add_compile_definitions(_GLIBCXX_USE_CXX11_ABI=0 google=ascend_private)

include_directories(${ASCEND_INSTALL_PATH})
include_directories(${ASCEND_INSTALL_PATH}/include)
include_directories(${CMAKE_CURRENT_SOURCE_DIR})
set(CMAKE_PREFIX_PATH ${ASCEND_INSTALL_PATH}/lib64 ${CMAKE_PREFIX_PATH})

# ES 构图 / 图 / 安全函数
find_library(ES_MATH NAMES es_math REQUIRED)
find_library(GRAPH NAMES graph REQUIRED)
find_library(GRAPH_BASE NAMES graph_base REQUIRED)
find_library(EAGER_STYLE_GRAPH_BUILDER_BASE NAMES eager_style_graph_builder_base REQUIRED)
find_library(C_SEC NAMES c_sec REQUIRED)
# 离线编译 / 运行时
find_library(GE_COMPILE NAMES ge_compiler REQUIRED)
find_library(GE_RUNNER NAMES ge_runner REQUIRED)
find_library(ACL_RT NAMES acl_rt REQUIRED)
find_library(ACL_MDL NAMES acl_mdl REQUIRED)

set(GRAPH_LIBS ${ES_MATH} ${GRAPH} ${GRAPH_BASE} ${EAGER_STYLE_GRAPH_BUILDER_BASE} ${C_SEC})
set(LINK_LIBS ${GRAPH_LIBS} ${GE_COMPILE} ${GE_RUNNER} ${ACL_RT} ${ACL_MDL})

add_executable(build_add_model build_add_model.cpp common.cpp)
target_link_options(build_add_model PRIVATE -Wl,--no-as-needed)
target_link_libraries(build_add_model PRIVATE ${LINK_LIBS})

add_executable(run_add_model run_add_model.cpp common.cpp)
target_link_options(run_add_model PRIVATE -Wl,--no-as-needed)
target_link_libraries(run_add_model PRIVATE ${LINK_LIBS})


### 7.5 在昇腾环境编译并运行

7.1–7.4 已经把 `common.h` / `common.cpp`、`build_add_model.cpp`、`run_add_model.cpp` 和 `CMakeLists.txt` 落盘到 `Sources/01.04`。下面**直接 CMake 编译这套工程并执行**——先用 `build_add_model` 生成 `add_sample.om`，再用 `run_add_model` 加载推理。

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p><strong>需在已安装 CANN（toolkit + ops）的昇腾环境执行，并已 <code>source</code> CANN 的 <code>set_env.sh</code>（通常是 <code>/usr/local/Ascend/ascend-toolkit/set_env.sh</code>，使 <code>ASCEND_HOME_PATH</code> 与库路径生效）。</strong> 轻量 notebook 环境无 NPU/CANN 时，CMake <code>find_library</code> 会找不到昇腾库而报错，属正常现象。</p>
<p>无卡仅编译时给 <code>build_add_model</code> 传目标芯片：<code>./build/build_add_model --soc-version Ascend910B1</code>。</p>
</blockquote>
</div>

执行成功会打印推理结果，应与第 1 个 numpy cell 的 `expected` 一致：

```text
[Info] 模型推理结果:
Output[1] shape: 2 3
 data:
 11 22 33
 44 55 66
```

下面的 cell 直接编译并运行本节落盘的工程：

In [ ]:
# === 需昇腾环境：用本节落盘的 C++ 源码编译并执行 ===
# 轻量 notebook 无 CANN/NPU 时，cmake/find_library 找不到昇腾库会报错，属正常现象。
# 昇腾环境下（已 source set_env）应依次打印 [Info] 模型保存成功 ... 和 Output[1] ... 11 22 33 / 44 55 66
!cd Sources/01.04 && cmake -B build -S . && cmake --build build -j && ./build/build_add_model && ./build/run_add_model

### 7.6 完整工程与参考实现

上面 7.1–7.5 已经手写并跑通了 C++ 全流程。GE 仓库还提供了封装好环境变量与依赖的**完整工程**（含 bundle 多模型等进阶场景），C++ 与 Python 两种语言都有：

- **C++**：[examples/offline_compile_run/cpp](https://gitcode.com/cann/ge/tree/master/examples/offline_compile_run/cpp)，一键 `bash run_sample.sh -t sample_and_run`（无卡加 `-s Ascend910B1`）。
- **Python**：[examples/offline_compile_run/python](https://gitcode.com/cann/ge/tree/master/examples/offline_compile_run/python)，一键 `bash run_sample.sh -t sample_and_run_python`。Python 侧 API 与 C++ 一一对应：`ge.es.graph_builder.GraphBuilder` ↔ `ge::es::EsGraphBuilder`，`ge.offline_compile.build_*` ↔ `ge::aclgrphBuild*`，`acl.mdl.execute` ↔ `aclmdlExecute`。

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>两种语言的样例输入与期望输出完全相同（<code>[[11,22,33],[44,55,66]]</code>），可任选其一上手。</p>
</blockquote>
</div>

## 课后练习

本节讲了 ATC 离线编译四步、ES 构图、OM 产物、ACL 加载执行。请完成以下题目自测。

1. （判断题）ATC 编译 OM 模型时需要昇腾设备在线才能完成编译。

2. （判断题）`aclmdlExecute` 是同步执行接口，调用返回时推理已完成。

3. （单选题）以下哪个接口用于将 Graph 编译为离线模型并保存到内存缓冲区？
    A. `aclgrphBuildInitialize`
    B. `aclgrphBuildModel`
    C. `aclgrphSaveModel`
    D. `aclgrphBuildFinalize`

4. （单选题）OM 离线模型中不包含以下哪项内容？
    A. 算子二进制
    B. 权重数据
    C. 原始模型文件（.onnx / .pb）
    D. 执行流信息

5. （多选题）以下关于 ACL 模型加载接口的描述，哪些是正确的？
    A. `aclmdlLoadFromFile` 从文件路径加载 OM 模型
    B. `aclmdlLoadFromMem` 从内存缓冲区加载 OM 模型
    C. `aclmdlLoadFromFileWithMem` 由用户指定工作/权重内存，便于精细化内存管理
    D. 所有加载接口都会返回 model_id，后续通过此 ID 操作模型

6. （单选题）`atc` 命令中，`--framework` 取值 **5** 代表的原始框架是？
    A. TensorFlow
    B. Caffe
    C. ONNX
    D. MindSpore air

7. （填空/简答题）本节 Add 样例用 ES 构图（`ge::es::EsGraphBuilder`）：用 `graph_builder->CreateInput(...)` 创建输入，用 `____` 设置图的输出，最后用 `____` 完成构图并产出 `ge::Graph`。

8. （简答题）给定输入 `x=[[1,2,3],[4,5,6]]`、`y=[[10,20,30],[40,50,60]]`，写出 Add 图的期望输出，并说明为什么 numpy 对拍 cell 能在无 NPU 环境验证它。

9. （判断题）异步执行 `aclmdlExecuteAsync` 调用返回后，可以立即从输出 dataset 读到正确结果，无需额外同步。

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/02.02_answer.txt